In [1]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [2]:
# pip install transformers sentence-transformers scikit-learn requests trafilatura nltk


In [3]:
# pip install lxml_html_clean


In [4]:
import nltk
nltk.download('vader_lexicon')


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\aratb\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [5]:
import io
import math
import time
import collections
from urllib.parse import urlparse

from tqdm import tqdm
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import requests
import trafilatura

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1 - Clean News Content Data Prep

In [6]:
SENTIMENT_RAW = "s3://ozu-ml-research/realestate_multimodal_ml/sentiment/bq-export-gdelt.csv"

bucket = "ozu-ml-research"
key = "realestate_multimodal_ml/sentiment/bq-export-gdelt.csv"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
meta = s3.head_object(Bucket=bucket, Key=key)
total_bytes = meta["ContentLength"]

buffer = io.BytesIO()
with tqdm(total=total_bytes, unit="B", unit_scale=True, desc="Downloading sentiment CSV") as pbar:
    s3.download_fileobj(Bucket=bucket, Key=key, Fileobj=buffer, Callback=lambda b: pbar.update(b))

buffer.seek(0)
df = pd.read_csv(buffer)
print(f"Loaded {len(df)} sentiment rows from {SENTIMENT_RAW}")


Loaded 314431 sentiment rows from s3://ozu-ml-research/realestate_multimodal_ml/sentiment/bq-export-gdelt.csv


In [7]:
english_sources = [
    "arabianbusiness", "dubaichronicle", "emirates247", "gulfbusiness", "gulfnews",
    "gulftoday", "khaleejtimes", "thearabianpost", "thenationalnews", "saudigazette",
    "english.alarabiya", "english.aawsat", "english.alriyadh", "english.asharq", "wam.ae/en",
    "bbc monitoring", "nytimes.com", "washingtonpost.com", "wsj.com", "usatoday.com",
    "latimes.com", "chicagotribune.com", "bostonglobe.com", "houstonchronicle.com",
    "abcnews.go.com", "cbsnews.com", "nbcnews.com", "pbs.org", "foxnews.com", "cnn.com",
    "msnbc.com", "apnews.com", "reuters.com", "bloomberg.com", "marketwatch.com", "barrons.com",
    "cnbc.com", "forbes.com", "businessinsider.com", "npr.org", "publicradioexchange.org",
    "theatlantic.com", "time.com", "newsweek.com", "newyorker.com", "economist.com",
    "motherjones.com", "politico.com", "vox.com", "huffpost.com", "axios.com",
    "buzzfeednews.com", "slate.com", "propublica.org", "fivethirtyeight.com", "thehill.com",
    "realclearpolitics.com", "miamiherald.com", "inquirer.com", "startribune.com",
    "seattletimes.com", "tampabay.com", "sfchronicle.com", "freep.com", "dallasnews.com",
    "ajc.com", "denverpost.com", "publicintegrity.org", "insideclimatenews.org", "khn.org",
    "politifact.com", "factcheck.org", "fortune.com", "fastcompany.com", "techcrunch.com",
    "theverge.com", "c-span.org", "cheddar.com"
]

arabic_only = [
    "albayan", "alittihad", "alkhaleej", "emaratalyoum", "aljarida", "alanba", "alarab",
    "alayam", "albiladpress", "almowaten", "alqabas", "alraimedia", "alroya", "madina",
    "marsd", "seyassah", "sharq", "alriyadh"
]

df = df[~df["clean_source"].astype(str).str.contains("|".join(arabic_only), case=False, na=False)]


In [8]:
"|".join(english_sources)

'arabianbusiness|dubaichronicle|emirates247|gulfbusiness|gulfnews|gulftoday|khaleejtimes|thearabianpost|thenationalnews|saudigazette|english.alarabiya|english.aawsat|english.alriyadh|english.asharq|wam.ae/en|bbc monitoring|nytimes.com|washingtonpost.com|wsj.com|usatoday.com|latimes.com|chicagotribune.com|bostonglobe.com|houstonchronicle.com|abcnews.go.com|cbsnews.com|nbcnews.com|pbs.org|foxnews.com|cnn.com|msnbc.com|apnews.com|reuters.com|bloomberg.com|marketwatch.com|barrons.com|cnbc.com|forbes.com|businessinsider.com|npr.org|publicradioexchange.org|theatlantic.com|time.com|newsweek.com|newyorker.com|economist.com|motherjones.com|politico.com|vox.com|huffpost.com|axios.com|buzzfeednews.com|slate.com|propublica.org|fivethirtyeight.com|thehill.com|realclearpolitics.com|miamiherald.com|inquirer.com|startribune.com|seattletimes.com|tampabay.com|sfchronicle.com|freep.com|dallasnews.com|ajc.com|denverpost.com|publicintegrity.org|insideclimatenews.org|khn.org|politifact.com|factcheck.org|for

In [9]:
pd.Series(['arabianbusiness']).str.contains('arabianbusiness|dubaichronicle', regex=False, case=False, na=False)

0    False
dtype: bool

In [10]:
mask_english = df["clean_source"].astype(str).str.contains(
    "|".join(english_sources), case=False, na=False
)
df = df[mask_english].copy()
print(f"{len(df)} English-language rows retained.")

df = df[df['ECON_HOUSING_PRICES']]
print(f"{len(df)} rows after filtering for ECON_HOUSING_PRICES=True.")

df["Timestamp"] = pd.to_datetime(df["Timestamp"], utc=True, errors="coerce") # already in UTC from source
df = df.dropna(subset=["Timestamp"])
display(df.sort_values('Timestamp').head(2))

df = df.dropna(subset=["clean_link"]).drop_duplicates(subset=["clean_link"])
print(f"{len(df)} rows after dropping duplicates and missing links.")

192663 English-language rows retained.
37626 rows after filtering for ECON_HOUSING_PRICES=True.


,raw_extras,clean_source,clean_image,clean_title,clean_link,Timestamp,clean_tone,positive_score,negative_score,polarity,...,ECON_ENTREPRENEURSHIP,ECON_INTEREST_RATES,ECON_WORLDCURRENCIES_DOLLAR,ECON_WORLDCURRENCIES_EURO,ECON_CURRENCY_EXCHANGE_RATE,ECON_OILPRICE,ECON_BOYCOTT,ECON_BANKRUPTCY,ECON_ELECTRICALGENERATION,United Arab Emirates
158211,NaN,emirates247.com,http://cdn-wac.emirates247.com/polopoly_fs/1.5...,NaN,http://www.emirates247.com/news/emirates/hundr...,2015-04-01 03:15:00+00:00,4.142012,4.733728,0.591716,5.325444,...,False,False,False,False,False,False,False,False,False,True
272488,NaN,emirates247.com,http://cdn-wac.emirates247.com/polopoly_fs/1.5...,NaN,http://www.emirates247.com/news/emirates/11-pr...,2015-04-01 03:30:00+00:00,1.239669,2.892562,1.652893,4.545455,...,False,False,False,False,False,False,False,False,False,True


37582 rows after dropping duplicates and missing links.


In [11]:
import concurrent.futures
import urllib


NUM_BATCHES = 12
MAX_WORKERS = 32
PER_DOMAIN_DELAY = 1.0
REQUEST_TIMEOUT = 10
ROBOTS_TIMEOUT = 5
HEADERS = {"User-Agent": "AcademicSentimentBot/0.1 (+contact)"}

robots_cache = {}
last_hit = collections.defaultdict(float)

def allowed(url: str) -> bool:
    dom = urlparse(url).netloc
    rp = robots_cache.get(dom)
    if rp is None:
        rp = urllib.robotparser.RobotFileParser()
        try:
            r = requests.get(f"https://{dom}/robots.txt", headers=HEADERS, timeout=ROBOTS_TIMEOUT)
            rp.parse(r.text.splitlines())
        except Exception:
            pass
        robots_cache[dom] = rp
    return rp.can_fetch("*", url) if rp else True

def fetch_html(sess: requests.Session, url: str) -> str | None:
    dom = urlparse(url).netloc
    wait = PER_DOMAIN_DELAY - (time.time() - last_hit[dom])
    if wait > 0:
        time.sleep(wait)
    last_hit[dom] = time.time()
    try:
        r = sess.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        if r.status_code == 200 and "text/html" in r.headers.get("Content-Type", ""):
            return r.text
    except Exception:
        pass
    return None

def extract_title_and_text(html_text: str, url: str) -> tuple[str | None, str | None]:
    try:
        meta = trafilatura.extract_metadata(html_text, url=url)
    except Exception:
        meta = None
    title = meta.title if meta and getattr(meta, "title", None) else None
    try:
        text = trafilatura.extract(html_text, include_comments=False, include_tables=False)
    except Exception:
        text = None
    return title, text

def worker(url: str):
    if not hasattr(worker, "sess"):
        worker.sess = requests.Session()
    if not allowed(url):
        return url, None, None
    html_text = fetch_html(worker.sess, url)
    if not html_text:
        return url, None, None
    title, text = extract_title_and_text(html_text, url)
    return url, title, text

def run_crawl(df: pd.DataFrame, english_sources: list[str] | None = None) -> pd.DataFrame:
    if "clean_link" not in df.columns:
        raise ValueError("DataFrame must include a 'clean_link' column.")
    if english_sources and "clean_source" in df.columns:
        mask_eng = df["clean_source"].astype(str).str.contains("|".join(english_sources), case=False, na=False)
        df_use = df[mask_eng].copy()
    else:
        df_use = df.copy()

    urls = df_use["clean_link"].dropna().astype(str).unique().tolist()
    if not urls:
        df_out = df.copy()
        if "article_text" not in df_out.columns:
            df_out["article_text"] = None
        if "article_title_fetched" not in df_out.columns:
            df_out["article_title_fetched"] = None
        return df_out

    rng = np.random.default_rng(42)
    rng.shuffle(urls)
    chunk_len = math.ceil(len(urls) / NUM_BATCHES)

    batch_titles = {}
    batch_texts = {}

    for batch_no in range(1, NUM_BATCHES + 1):
        start = (batch_no - 1) * chunk_len
        stop = min(start + chunk_len, len(urls))
        batch_urls = urls[start:stop]
        if not batch_urls:
            break

        with concurrent.futures.ThreadPoolExecutor(MAX_WORKERS) as pool:
            futures = {pool.submit(worker, u): u for u in batch_urls}
            with tqdm(total=len(batch_urls), desc=f"Batch {batch_no}/{NUM_BATCHES}") as bar:
                while futures:
                    done, _ = concurrent.futures.wait(
                        futures, timeout=1, return_when=concurrent.futures.FIRST_COMPLETED
                    )
                    for fut in done:
                        u, title, txt = fut.result()
                        if txt:
                            batch_texts[u] = txt
                            batch_titles[u] = title
                        bar.update(1)
                        bar.set_postfix(success=len(batch_texts))
                        del futures[fut]

    df_out = df.copy()
    if "article_text" not in df_out.columns:
        df_out["article_text"] = None
    if "article_title_fetched" not in df_out.columns:
        df_out["article_title_fetched"] = None

    if batch_texts:
        mask_hits = df_out["clean_link"].astype(str).isin(batch_texts.keys())
        df_out.loc[mask_hits, "article_text"] = df_out.loc[mask_hits, "clean_link"].map(batch_texts)
        df_out.loc[mask_hits, "article_title_fetched"] = df_out.loc[mask_hits, "clean_link"].map(batch_titles).fillna("")

    return df_out


In [12]:
df = run_crawl(df)


Batch 12/12: 100%|██████████| 3130/3130 [03:27<00:00, 15.08it/s, success=25998]


### save the checkpoint and do not repeat the initial data cleaning after first run

In [ ]:
df.to_csv("gdelt_sentiment_with_text_checpoint.csv", index=False)